# 03 — Regression Model: Predicting Median Home Sales Price
**Project:** Charlotte NPA Housing Affordability Analysis  
**Target Variable:** `log_home_sales_price` (log of 2023 median home sales price)  
**Working dataset:** 444 NPAs from `npa_features_model`

## Findings Summary (read this first)

This notebook documents our regression modeling effort to predict `log(home_sales_price)` using 8 NPA-level demographic and economic features. Three model variants were compared (Full OLS, Reduced OLS, KNN) on a single 80/20 train-test split, then the winning specification was subjected to a multi-seed stability check across 10 random splits.

### Key finding: the regression model is unstable.
Across 10 random train-test splits, test R² ranged from **-0.18 to 0.28**, with a mean of **0.06** and a standard deviation of **0.14**. This indicates the model is not reliably learning a generalizable pattern from the available features. The single-split test R² of 0.24 produced by `random_state=42` is therefore not a representative result. On 4 of 10 seeds the model performs worse than predicting the mean.

### Why this matters:
NPA-level demographic and economic features alone are insufficient to reliably predict median home sales price in Charlotte. Reliable price prediction at the neighborhood level would require parcel-level housing characteristics (year built, square footage of comparable sales, transaction volume, school assignment, lot size) or finer-grained data than the Quality of Life Explorer provides at the NPA level.

We report this finding honestly rather than overclaiming based on a single lucky split. This is itself a meaningful negative result: it tells policymakers that the publicly available neighborhood-level data is not sufficient on its own for reliable price modeling, and that classification of displacement risk is a more tractable task with the same data.

### What this informs:
The classification model in **notebook 04** predicts `displacement_risk` as a binary label and is fundamentally an easier task. It is expected to produce more stable and useful results. The regression notebook stands as documentation of what NPA-level features cannot do.

### Why log target
EDA showed the raw target had skew = 2.988 and kurtosis = 13.73. The log-transformed target has skew = 0.898, which is in the acceptable range for OLS. We model on the log scale and back-transform predictions for interpretability.

### Process improvements made during this notebook
An initial run produced a numerically unstable model with a condition number of 2.14e+06 and CV R² that did not match train or test R². Three fixes were applied:

1. **All OLS models are now wrapped in a StandardScaler pipeline.** Mixing features at vastly different scales was the root cause of the multicollinearity warning and the unstable coefficients. Standardizing put every feature on a unit-variance scale so the matrix inversion is well-conditioned. After this fix the condition number dropped from 2.14e+06 to 6.31.
2. **Engineered features (`rent_to_income_ratio`, `income_per_sqft`) were dropped.** They are derived from features already in the model, so they added redundancy without adding signal.
3. **A multi-seed stability check** was added (section 7). This is the diagnostic that revealed the model's instability. Without it the project would have shipped the misleading single-split R² of 0.24.

## Notebook structure
1. Preflight check
2. Load model-ready table
3. Define feature sets
4. Train/test split with median imputation (fit on train only)
5. Reusable evaluation function
6. Fit and compare three model variants
7. Multi-seed stability check (the key diagnostic)
8. Refit winner with statsmodels for inference
9. Coefficient interpretation
10. Residual diagnostics
11. Back-transform to dollar scale
12. Save outputs to database

In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

base = os.path.abspath(os.path.join(os.getcwd(), '..'))
db_path = os.path.join(base, 'data', 'charlotte_housing.db')
print(f'Project root: {base}')
print(f'Database:     {db_path}')

## 1. Preflight Check

In [ ]:
REQUIRED_TABLE = 'npa_features_model'
REQUIRED_COLUMNS = [
    'npa_id', 'home_sales_price', 'log_home_sales_price', 'displacement_risk',
    'tree_canopy_pct', 'bachelors_pct', 'water_consumption_gpd', 'median_rent',
    'absenteeism_pct', 'household_income', 'housing_size_sqft', 'voter_participation_pct'
]
MIN_EXPECTED_ROWS = 400

def preflight():
    if not os.path.exists(db_path):
        raise FileNotFoundError(
            f'Database not found at {db_path}.\n'
            f'Fix: run notebook 01 to create the database.'
        )
    print(f'[OK] Database file exists ({os.path.getsize(db_path) / 1024:.1f} KB)')

    conn = sqlite3.connect(db_path)
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'].tolist()
    print(f'[OK] Tables in database: {tables}')

    if REQUIRED_TABLE not in tables:
        conn.close()
        raise RuntimeError(
            f'\nMissing required table: "{REQUIRED_TABLE}"\n'
            f'Fix: run notebook 02 end to end.'
        )
    print(f'[OK] Required table "{REQUIRED_TABLE}" exists')

    df_check = pd.read_sql(f'SELECT * FROM {REQUIRED_TABLE}', conn)
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df_check.columns]
    if missing_cols:
        conn.close()
        raise RuntimeError(f'\nMissing columns: {missing_cols}')
    print(f'[OK] All {len(REQUIRED_COLUMNS)} required columns present')

    if len(df_check) < MIN_EXPECTED_ROWS:
        conn.close()
        raise RuntimeError(f'\nRow count too low: {len(df_check)}')
    print(f'[OK] Row count: {len(df_check)} NPAs')

    if df_check['log_home_sales_price'].isnull().sum() > 0:
        conn.close()
        raise RuntimeError(f'\nNulls found in target column.')
    print(f'[OK] No nulls in target column')

    if sorted(df_check['displacement_risk'].dropna().unique().tolist()) != [0, 1]:
        conn.close()
        raise RuntimeError(f'\ndisplacement_risk not binary.')
    print(f'[OK] displacement_risk is binary {{0, 1}}')

    print('\nPreflight passed. Safe to proceed.')
    return conn

conn = preflight()

## 2. Load model-ready table

In [ ]:
df = pd.read_sql(f'SELECT * FROM {REQUIRED_TABLE}', conn)
print(f'Shape: {df.shape}')
df.head()

## 3. Define feature sets
Two sets. Engineered features dropped because they added multicollinearity without improving test R².

In [ ]:
features_full = [
    'tree_canopy_pct', 'bachelors_pct', 'water_consumption_gpd', 'median_rent',
    'absenteeism_pct', 'household_income', 'housing_size_sqft', 'voter_participation_pct'
]

features_reduced = [
    'bachelors_pct', 'water_consumption_gpd',
    'absenteeism_pct', 'household_income', 'housing_size_sqft'
]

for name, feats in [('full', features_full), ('reduced', features_reduced)]:
    missing = [f for f in feats if f not in df.columns]
    if missing:
        raise RuntimeError(f'Feature set "{name}" references missing columns: {missing}')
    print(f'Model {name}: {len(feats)} features (all present)')

## 4. Train/test split with median imputation

In [ ]:
y = df['log_home_sales_price']
X = df[features_full].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Median imputation fit on train only
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

test_npa_ids = df.loc[X_test.index, 'npa_id'].values

assert X_train.isnull().sum().sum() == 0
assert X_test.isnull().sum().sum() == 0

print(f'Train shape: {X_train.shape}')
print(f'Test shape:  {X_test.shape}')

## 5. Reusable evaluation function (with scaling for OLS)

In [ ]:
def make_ols_pipe():
    return Pipeline([
        ('scale', StandardScaler()),
        ('ols', LinearRegression())
    ])

def make_knn_pipe(k=10):
    return Pipeline([
        ('scale', StandardScaler()),
        ('knn', KNeighborsRegressor(n_neighbors=k))
    ])

def evaluate_model(pipe, feature_list, name, X_train, y_train, X_test, y_test, cv_folds=5):
    Xtr = X_train[feature_list]
    Xte = X_test[feature_list]

    kf = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipe, Xtr, y_train, cv=kf, scoring='r2')

    pipe.fit(Xtr, y_train)
    train_pred = pipe.predict(Xtr)
    test_pred = pipe.predict(Xte)

    return {
        'model': name,
        'n_features': len(feature_list),
        'train_r2': r2_score(y_train, train_pred),
        'cv_r2': cv_scores.mean(),
        'cv_r2_std': cv_scores.std(),
        'test_r2': r2_score(y_test, test_pred),
        'train_rmse': np.sqrt(mean_squared_error(y_train, train_pred)),
        'test_rmse': np.sqrt(mean_squared_error(y_test, test_pred)),
        'overfit_gap': r2_score(y_train, train_pred) - r2_score(y_test, test_pred),
    }

## 6. Fit and compare three candidate models
**Note:** the comparison table below shows results for a single train-test split (`random_state=42`). The stability check in section 7 reveals these results are not representative.

In [ ]:
results = []

results.append(evaluate_model(
    make_ols_pipe(), features_full, 'A — Full OLS scaled (8 features)',
    X_train, y_train, X_test, y_test
))

results.append(evaluate_model(
    make_ols_pipe(), features_reduced, 'B — Reduced OLS scaled (5 features)',
    X_train, y_train, X_test, y_test
))

results.append(evaluate_model(
    make_knn_pipe(k=10), features_full, 'C — KNN k=10 scaled (8 features)',
    X_train, y_train, X_test, y_test
))

results_df = pd.DataFrame(results)
display_cols = ['model', 'n_features', 'train_r2', 'cv_r2', 'cv_r2_std', 'test_r2',
                'train_rmse', 'test_rmse', 'overfit_gap']
display_df = results_df[display_cols].copy()
for col in display_df.columns[2:]:
    display_df[col] = display_df[col].round(4)
display_df.sort_values('test_r2', ascending=False).reset_index(drop=True)

In [ ]:
for r in results:
    if r['test_r2'] < 0:
        print(f"WARNING: {r['model']} has negative test R² ({r['test_r2']:.3f})")
    if r['overfit_gap'] > 0.15:
        print(f"WARNING: {r['model']} has large overfit gap ({r['overfit_gap']:.3f})")
    if r['overfit_gap'] < -0.10:
        print(f"WARNING: {r['model']} has negative overfit gap ({r['overfit_gap']:.3f}). "
              f"Test outperforms train, suggesting a lucky split or small-sample variance.")
    if r['cv_r2_std'] > 0.10:
        print(f"WARNING: {r['model']} has high CV variance (std={r['cv_r2_std']:.3f})")
    if abs(r['cv_r2'] - r['test_r2']) > 0.15:
        print(f"WARNING: {r['model']} has large CV-test gap (CV={r['cv_r2']:.3f}, test={r['test_r2']:.3f})")

print('\nIf no warnings printed, all models trained without obvious issues.')

## 7. Multi-seed stability check (KEY DIAGNOSTIC)
The negative overfit gap and large CV-test gap warnings above suggested the single-split result might not be representative. We refit the winning OLS specification across 10 different random_state values and report the spread of test R².

**This is the diagnostic that determines whether the regression model is a real result or noise.** A stable model should produce similar test R² values regardless of which random split is used. Large variance across seeds means the model is fitting noise.

In [ ]:
ols_results = [r for r in results if 'OLS' in r['model']]
best_ols = max(ols_results, key=lambda r: r['test_r2'])
feat_lookup = {
    'A — Full OLS scaled (8 features)': features_full,
    'B — Reduced OLS scaled (5 features)': features_reduced,
}
best_features = feat_lookup[best_ols['model']]

print(f'Running stability check on: {best_ols["model"]}')
print(f'Features: {best_features}\n')

stability_rows = []
for seed in range(10):
    Xs = df[best_features].copy()
    ys = df['log_home_sales_price']
    Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(Xs, ys, test_size=0.20, random_state=seed)
    medians = Xs_tr.median()
    Xs_tr = Xs_tr.fillna(medians)
    Xs_te = Xs_te.fillna(medians)

    pipe = make_ols_pipe()
    pipe.fit(Xs_tr, ys_tr)
    train_r2 = r2_score(ys_tr, pipe.predict(Xs_tr))
    test_r2 = r2_score(ys_te, pipe.predict(Xs_te))
    stability_rows.append({'seed': seed, 'train_r2': train_r2, 'test_r2': test_r2,
                            'overfit_gap': train_r2 - test_r2})

stab_df = pd.DataFrame(stability_rows).round(4)
print(stab_df.to_string(index=False))
print(f'\nTest R² across 10 seeds:')
print(f'  Mean: {stab_df["test_r2"].mean():.4f}')
print(f'  Std:  {stab_df["test_r2"].std():.4f}')
print(f'  Min:  {stab_df["test_r2"].min():.4f}')
print(f'  Max:  {stab_df["test_r2"].max():.4f}')
print(f'  Range: {stab_df["test_r2"].max() - stab_df["test_r2"].min():.4f}')

# Verdict
test_std = stab_df['test_r2'].std()
if test_std < 0.05:
    print('\nVERDICT: Model is STABLE across random splits. Result is reliable.')
elif test_std < 0.10:
    print('\nVERDICT: Model is MODERATELY stable. Report with limitations.')
else:
    print('\nVERDICT: Model is UNSTABLE. The single-split result is not representative.')
    print('Interpret as: NPA-level features are insufficient to reliably predict home prices.')
    print('See findings summary at the top of this notebook.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = stab_df['seed']
ax.plot(x, stab_df['train_r2'], 'o-', label='Train R²', color='steelblue')
ax.plot(x, stab_df['test_r2'], 's-', label='Test R²', color='firebrick')
ax.axhline(stab_df['test_r2'].mean(), color='firebrick', linestyle=':', alpha=0.5,
           label=f'Mean test R² = {stab_df["test_r2"].mean():.3f}')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('random_state seed')
ax.set_ylabel('R²')
ax.set_title('Stability check: train and test R² across 10 random splits')
ax.legend()
ax.set_xticks(x)
plt.tight_layout()
plt.show()

## 8. Refit the winning OLS with statsmodels
Even though the model is unstable across splits, we still inspect the coefficients on the `random_state=42` fit to understand which features show the most signal when the model does fit. We standardize the features manually here so coefficients represent the effect of a 1 standard deviation change.

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train[best_features]),
    columns=best_features, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test[best_features]),
    columns=best_features, index=X_test.index
)

X_train_sm = sm.add_constant(X_train_scaled)
X_test_sm = sm.add_constant(X_test_scaled)

ols_model = sm.OLS(y_train, X_train_sm).fit()
print(ols_model.summary())

In [ ]:
print(f'Condition number: {ols_model.condition_number:.2f}')
if ols_model.condition_number < 30:
    print('GOOD: condition number is low. Multicollinearity is not a problem.')
elif ols_model.condition_number < 100:
    print('OK: condition number is moderate.')
else:
    print('WARNING: condition number is still high. Multicollinearity may remain.')

## 9. Coefficient interpretation
Coefficients are on standardized features, so each one represents the effect of a 1 standard deviation increase in that feature on log(price). To convert to a percent effect on price: `(exp(coef) - 1) × 100`%.

**Caveat:** because the model is unstable across random splits, these specific coefficient values reflect one particular fit. The signs and rough magnitudes may be informative about which features have the strongest signal, but the exact values should not be treated as precise estimates.

In [ ]:
coef_table = pd.DataFrame({
    'feature': ols_model.params.index,
    'coef_std_units': ols_model.params.values,
    'std_err': ols_model.bse.values,
    'p_value': ols_model.pvalues.values,
    'pct_effect_per_1sd': (np.exp(ols_model.params.values) - 1) * 100,
    'significant_05': ols_model.pvalues.values < 0.05,
}).round(4)
coef_table

In [ ]:
plot_df = coef_table[coef_table['feature'] != 'const'].copy()
plot_df = plot_df.sort_values('coef_std_units', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if s else 'lightgray' for s in plot_df['significant_05']]
ax.barh(plot_df['feature'], plot_df['coef_std_units'], color=colors, edgecolor='black')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (standardized log-dollar units)')
ax.set_title('OLS coefficients — effect of a 1 SD increase in each feature\n(blue = significant at p<0.05, gray = not significant)')
plt.tight_layout()
plt.show()

## 10. Residual diagnostics

In [ ]:
y_test_pred = ols_model.predict(X_test_sm)
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_test_pred, residuals, alpha=0.5, edgecolor='black')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted values (log price)')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted (test set)')

sm.qqplot(residuals, line='s', ax=axes[1])
axes[1].set_title('QQ plot of residuals')

axes[2].scatter(y_test, y_test_pred, alpha=0.5, edgecolor='black')
lo, hi = y_test.min(), y_test.max()
axes[2].plot([lo, hi], [lo, hi], 'r--', linewidth=2)
axes[2].set_xlabel('Actual log price')
axes[2].set_ylabel('Predicted log price')
axes[2].set_title(f'Actual vs Predicted (test R² = {best_ols["test_r2"]:.3f})')

plt.tight_layout()
plt.show()

## 11. Back-transform to dollar scale

In [ ]:
y_test_pred_dollars = np.exp(y_test_pred)
y_test_dollars = np.exp(y_test)

dollar_rmse = np.sqrt(mean_squared_error(y_test_dollars, y_test_pred_dollars))
dollar_mae = np.mean(np.abs(y_test_dollars.values - y_test_pred_dollars.values))
median_actual = y_test_dollars.median()

print(f'Test RMSE (dollar scale): ${dollar_rmse:,.0f}')
print(f'Test MAE (dollar scale):  ${dollar_mae:,.0f}')
print(f'Median actual test price: ${median_actual:,.0f}')
print(f'Median absolute error as percent of median actual: {dollar_mae / median_actual * 100:.1f}%')

In [ ]:
pred_df = pd.DataFrame({
    'npa_id': test_npa_ids,
    'actual_price': y_test_dollars.values,
    'predicted_price': y_test_pred_dollars.values,
    'residual_dollars': y_test_dollars.values - y_test_pred_dollars.values,
})
pred_df['abs_pct_error'] = (pred_df['residual_dollars'].abs() / pred_df['actual_price'] * 100).round(2)
pred_df = pred_df.sort_values('abs_pct_error').reset_index(drop=True)

print('Best predictions (smallest % error):')
print(pred_df.head(5).to_string(index=False))
print('\nWorst predictions (largest % error):')
print(pred_df.tail(5).to_string(index=False))

## 12. Save outputs to database

In [ ]:
results_df.to_sql('regression_model_comparison', conn, if_exists='replace', index=False)
pred_df.to_sql('regression_test_predictions', conn, if_exists='replace', index=False)
coef_table.to_sql('regression_final_coefficients', conn, if_exists='replace', index=False)
stab_df.to_sql('regression_stability_check', conn, if_exists='replace', index=False)

for tbl in ['regression_model_comparison', 'regression_test_predictions',
            'regression_final_coefficients', 'regression_stability_check']:
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {tbl}', conn)['n'][0]
    print(f'{tbl}: {n} rows saved')

In [ ]:
conn.close()
print('\nConnection closed. Regression notebook complete.')
print('\nNext: notebook 04 (classification of displacement_risk).')